# Syndicate Alpha — High-Speed Databento Batch Downloader (2-Year OHLCV + 5-Month MBP-10)
Uses Databento's official **`batch.submit_job`** + **`batch.download`** architecture to process 5 full months of CME COMEX Gold (`GC.v.0`) **Level 2 MBP-10** order book data on Databento's cloud cluster and store straight into your 5 TB Google Drive (**0 GB used on your PC**).

**OHLCV Mode:** Auto-detects your existing OHLCV file in Google Drive and locks the backtest window to the last **2 Years** (2024-2026).

In [ ]:
# =====================================================================
# STEP 1: Install Databento & Connect Your 5 TB Google Drive
# =====================================================================
!pip install -q databento pyarrow pandas numpy

import os
from google.colab import drive

drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/Syndicate_Alpha_Gold_Data'
RAW_OHLCV_DIR = os.path.join(BASE_DIR, 'raw_ohlcv_1m')
RAW_MBP10_DIR = os.path.join(BASE_DIR, 'raw_mbp10_5months')
PROCESSED_DIR = os.path.join(BASE_DIR, 'processed_for_antigravity')

for d in [BASE_DIR, RAW_OHLCV_DIR, RAW_MBP10_DIR, PROCESSED_DIR]:
    os.makedirs(d, exist_ok=True)

print('SUCCESS: Google Drive connected! Storage folders ready at:', BASE_DIR)

In [ ]:
# =====================================================================
# STEP 2: Paste Your Databento API Key Below & Check Cost
# =====================================================================
import databento as db

# >>> PASTE YOUR KEY INSIDE THE QUOTES BELOW <<<
DATABENTO_API_KEY = "PASTE_YOUR_KEY_HERE"

DATASET = 'GLBX.MDP3'     # CME Globex COMEX Exchange
SYMBOL = 'GC.v.0'         # Continuous Front-Month Gold Futures by Volume
STYPE_IN = 'continuous'

# 5 Months of Level-2 MBP-10 Book Depth (June 1 to Nov 1)
MBP10_START = '2026-06-01'
MBP10_END   = '2026-11-01'

# 2-Year 1-Minute OHLCV Window (Nov 2024 to Nov 2026)
OHLCV_2YR_START = '2024-11-01'
OHLCV_2YR_END   = '2026-11-01'

client = db.Historical(key=DATABENTO_API_KEY)

cost_mbp10 = client.metadata.get_cost(
    dataset=DATASET, symbols=[SYMBOL], stype_in=STYPE_IN,
    schema='mbp-10', start=MBP10_START, end=MBP10_END
)
size_mbp10_gb = client.metadata.get_billable_size(
    dataset=DATASET, symbols=[SYMBOL], stype_in=STYPE_IN,
    schema='mbp-10', start=MBP10_START, end=MBP10_END
) / (1024**3)

print('=================================================================')
print(f'5-Month Level-2 MBP-10 ({MBP10_START} -> {MBP10_END}) : {size_mbp10_gb:.2f} GB')
print(f'Estimated Databento Cost                      : ${cost_mbp10:.2f} USD')
print('=================================================================')

In [ ]:
# =====================================================================
# STEP 3: Auto-Detect Existing OHLCV & High-Speed Batch Job for MBP-10
# =====================================================================
import glob
import time

# 3A: Auto-detect already downloaded OHLCV files in Google Drive
existing_ohlcv = glob.glob(os.path.join(BASE_DIR, '**/*ohlcv*.dbn.zst'), recursive=True)
if existing_ohlcv:
    print(f'[AUTO-DETECTED] Found existing OHLCV in Google Drive: {os.path.basename(existing_ohlcv[0])}')
    print('  -> Using this file! Will slice exact 2-Year window (2024-2026) for backtesting without re-downloading!')
else:
    ohlcv_target = os.path.join(RAW_OHLCV_DIR, f'GC_v0_ohlcv1m_{OHLCV_2YR_START}_to_{OHLCV_2YR_END}.dbn.zst')
    print(f'Downloading 2-Year 1M OHLCV ({OHLCV_2YR_START} to {OHLCV_2YR_END})...')
    client.timeseries.get_range(
        dataset=DATASET, symbols=[SYMBOL], stype_in=STYPE_IN,
        schema='ohlcv-1m', start=OHLCV_2YR_START, end=OHLCV_2YR_END,
        path=ohlcv_target
    )
    print('Saved 2-Year OHLCV to Google Drive.')

# 3B: Submit High-Speed Batch Job for 5 Months of MBP-10
print('\nSubmitting 5-Month MBP-10 Batch Job to Databento cluster...')
job = client.batch.submit_job(
    dataset=DATASET,
    symbols=[SYMBOL],
    stype_in=STYPE_IN,
    schema='mbp-10',
    start=MBP10_START,
    end=MBP10_END,
    encoding='dbn',
    compression='zstd',
    split_duration='week'
)

job_id = job['id']
print(f'SUCCESS: Batch Job Submitted! Job ID: {job_id}')
print('Waiting for Databento cloud servers to prepare files (typically 2-4 minutes)...')

# Wait for Databento's servers to finish packing the files
while True:
    details = client.batch.get_job_details(job_id)
    state = details.get('state', 'in_progress')
    print(f'  -> Databento Server State: {state} ...')
    if state == 'done':
        break
    time.sleep(15)

# Download the completed batch straight into Google Drive
print('\nDownloading completed files directly into your 5TB Google Drive...')
downloaded_files = client.batch.download(
    job_id=job_id,
    output_dir=RAW_MBP10_DIR
)

print(f'\nCOMPLETE! Successfully saved {len(downloaded_files)} MBP-10 files directly to your Google Drive!')

In [ ]:
# =====================================================================
# STEP 4: Build Master 2-Year Backtest Parquet File for Antigravity
# =====================================================================
import glob
import numpy as np
import pandas as pd

# 4A: Load and filter OHLCV to exact 2-Year window (2024 to 2026)
ohlcv_candidates = glob.glob(os.path.join(BASE_DIR, '**/*ohlcv*.dbn.zst'), recursive=True)
master_ohlcv = pd.DataFrame()
if ohlcv_candidates:
    print(f'Loading 2-Year OHLCV slice from: {os.path.basename(ohlcv_candidates[0])}')
    store = db.DBNStore.from_file(ohlcv_candidates[0])
    df_ohlcv = store.to_df()
    if not isinstance(df_ohlcv.index, pd.DatetimeIndex):
        df_ohlcv.index = pd.to_datetime(df_ohlcv.index, utc=True)
    # Slice exact 2-year window (2024-11-01 to 2026-11-01)
    master_ohlcv = df_ohlcv.loc['2024-11-01':'2026-11-01'][['open', 'high', 'low', 'close', 'volume']]
    print(f'Loaded 2-Year 1M Bars: {len(master_ohlcv):,} bars.')

# 4B: Process MBP-10 Level 2 Depth & CVD for the 5-Month Window
mbp_files = sorted(glob.glob(os.path.join(RAW_MBP10_DIR, '**/*.dbn.zst'), recursive=True))
print(f'\nFound {len(mbp_files)} MBP-10 files in Google Drive. Distilling L2 DOM & CVD...')

l2_bars = []
for f in mbp_files:
    print('  Processing L2:', os.path.basename(f))
    store = db.DBNStore.from_file(f)
    df = store.to_df()
    if df.empty:
        continue
    if not isinstance(df.index, pd.DatetimeIndex):
        df.index = pd.to_datetime(df.index, utc=True)

    bid_cols = [f'bid_sz_0{i}' for i in range(10) if f'bid_sz_0{i}' in df.columns]
    ask_cols = [f'ask_sz_0{i}' for i in range(10) if f'ask_sz_0{i}' in df.columns]
    df['bid_depth_10'] = df[bid_cols].sum(axis=1)
    df['ask_depth_10'] = df[ask_cols].sum(axis=1)
    df['l2_ratio'] = (df['bid_depth_10'] + 1.0) / (df['ask_depth_10'] + 1.0)
    df['spread_pts'] = (df['ask_px_00'] - df['bid_px_00']).clip(lower=0)

    trades = df[df['action'] == 'T'].copy() if 'action' in df.columns else df.copy()
    if trades.empty:
        continue
    trades['delta'] = np.where(trades['side'] == 'B', trades['size'], np.where(trades['side'] == 'A', -trades['size'], 0))

    b1m = trades['price'].resample('1min').ohlc()
    b1m['volume'] = trades['size'].resample('1min').sum()
    b1m['bar_delta'] = trades['delta'].resample('1min').sum()
    b1m['l2_bid_ask_ratio_mean'] = df['l2_ratio'].resample('1min').mean()
    b1m['l2_bid_ask_ratio_max'] = df['l2_ratio'].resample('1min').max()
    b1m['max_bid_wall_10'] = df['bid_depth_10'].resample('1min').max()
    b1m['max_ask_wall_10'] = df['ask_depth_10'].resample('1min').max()
    b1m['avg_spread_pts'] = df['spread_pts'].resample('1min').mean()
    l2_bars.append(b1m.dropna(subset=['close']))

if l2_bars:
    master_l2 = pd.concat(l2_bars).sort_index()
    master_l2 = master_l2[~master_l2.index.duplicated(keep='last')]
    master_l2['cvd'] = master_l2['bar_delta'].cumsum()
    
    # Combine 2-Year OHLCV with 5-Month L2/CVD
    final_dataset = master_ohlcv.combine_first(master_l2) if not master_ohlcv.empty else master_l2
    out_parquet = os.path.join(PROCESSED_DIR, 'GC_v0_2Year_L2_OrderFlow_Master.parquet')
    final_dataset.to_parquet(out_parquet, compression='zstd')
    print(f'\nREADY FOR ANTIGRAVITY! Saved 2-Year Master Backtest file to: {out_parquet}')
    print(f'Total Bars: {len(final_dataset):,}')